# Cheap-guardrails pilot (full): guardrails for small local tool-using agents on tau-bench retail
Self-contained. Needs **GPU T4 x2** and **Internet on**.

In [ ]:
# 0. Configuration
MODELS = ["qwen2.5:7b", "qwen2.5:3b"]
USER_MODEL = "qwen2.5:14b"
ARMS = ["A0", "A1", "A2"]
PRIMARY = [0, 2, 5, 6, 9, 12, 13, 18, 21, 26, 27, 28, 29, 35, 40, 43, 45, 47, 48, 49, 54, 63, 66, 69, 72, 74, 81, 83, 86, 89, 90, 95, 96, 98, 101, 102, 103, 106, 108, 112]
EXTENSION = [1, 3, 4, 7, 8, 10, 11, 14, 15, 16, 17, 19, 20, 22, 23, 24, 25, 30, 31, 32, 33, 34, 36, 37, 38, 39, 41, 42, 44, 46, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 64, 65, 67, 68, 70, 71, 73, 75, 76, 77, 78, 79, 80, 82, 84, 85, 87, 88, 91, 92, 93, 94, 97, 99, 100, 104, 105, 107, 109, 110, 111, 113, 114]
PLAN = [("qwen2.5:7b", PRIMARY), ("qwen2.5:3b", PRIMARY),
        ("qwen2.5:7b", EXTENSION), ("qwen2.5:3b", EXTENSION)]
BUDGET_S = int(10.5 * 3600)  # stop starting new episodes after 10.5 h


In [ ]:
# 1. Ollama + tau-bench (pinned) + vendored retail subset + harness
import os, subprocess, sys, time, json, shutil
W = "/kaggle/working"; os.chdir(W)
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(cmd)
    return r
sh("nvidia-smi -L")
sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null", check=False)
if shutil.which("ollama") is None:
    sh("curl -fsSL https://ollama.com/install.sh | sh", check=False)
if shutil.which("ollama") is None:
    raise RuntimeError("ollama install failed")
sh("ollama --version", check=False)
sh("pip -q install jsonschema==4.23.0", check=False)
if not os.path.isdir("tau-bench"):
    sh("git clone -q https://github.com/sierra-research/tau-bench.git")
sh("cd tau-bench && git checkout -q 59a200c6d575d595120f1cb70fea53cef0632f6b && git log -1 --format=%H")
open("vendor_tau.py", "w").write('"""Build a minimal, dependency-light copy of the tau-bench retail domain.\n\ntau-bench (Sierra, MIT licence) imports litellm at package import time. The pilot\nonly needs the retail tools, data, policy (wiki) and test tasks, so we copy those\nfiles into a local package `tbr` and rewrite the two internal imports.\nUsage: python -I vendor_tau.py <path-to-tau-bench-repo> <output-dir>\n"""\nimport os, re, shutil, sys\n\nTBTYPES = \'\'\'# Minimal stand-ins for the upstream Action / Task types (same fields).\nclass Action:\n    def __init__(self, name, kwargs):\n        self.name = name\n        self.kwargs = kwargs\n    def to_dict(self):\n        return {"name": self.name, "kwargs": self.kwargs}\n\n\nclass Task:\n    def __init__(self, user_id, actions, instruction, outputs, annotator=None):\n        self.user_id = user_id\n        self.actions = actions\n        self.instruction = instruction\n        self.outputs = outputs\n        self.annotator = annotator\n\'\'\'\n\n\ndef build(repo, out):\n    src = os.path.join(repo, "tau_bench")\n    pkg = os.path.join(out, "tbr")\n    if os.path.isdir(pkg):\n        shutil.rmtree(pkg)\n    os.makedirs(os.path.join(pkg, "tools"))\n    os.makedirs(os.path.join(pkg, "data"))\n    shutil.copy(os.path.join(repo, "LICENSE"), os.path.join(pkg, "LICENSE_tau_bench"))\n    shutil.copy(os.path.join(src, "envs", "tool.py"), os.path.join(pkg, "tool.py"))\n    # types: the tasks only need Action and Task; plain classes avoid pydantic\n    open(os.path.join(pkg, "tbtypes.py"), "w", encoding="utf-8").write(TBTYPES)\n    shutil.copy(os.path.join(src, "envs", "retail", "wiki.md"), os.path.join(pkg, "wiki.md"))\n    for f in ("orders.json", "products.json", "users.json", "__init__.py"):\n        shutil.copy(os.path.join(src, "envs", "retail", "data", f), os.path.join(pkg, "data", f))\n    tools_dir = os.path.join(src, "envs", "retail", "tools")\n    for f in os.listdir(tools_dir):\n        if f.endswith(".py"):\n            shutil.copy(os.path.join(tools_dir, f), os.path.join(pkg, "tools", f))\n    shutil.copy(os.path.join(src, "envs", "retail", "tasks_test.py"), os.path.join(pkg, "tasks_test.py"))\n    open(os.path.join(pkg, "__init__.py"), "w").write("# vendored subset of tau-bench retail (MIT, Sierra)\\n")\n    n = 0\n    for root, _, files in os.walk(pkg):\n        for f in files:\n            if not f.endswith(".py"):\n                continue\n            p = os.path.join(root, f)\n            s = open(p, encoding="utf-8").read()\n            s2 = s.replace("from tau_bench.envs.tool import Tool", "from tbr.tool import Tool")\n            s2 = s2.replace("from tau_bench.types import", "from tbr.tbtypes import")\n            if "tau_bench" in s2:\n                raise SystemExit(f"unexpected tau_bench import left in {p}")\n            if s2 != s:\n                n += 1\n                open(p, "w", encoding="utf-8").write(s2)\n    return n\n\nif __name__ == "__main__":\n    n = build(sys.argv[1], sys.argv[2])\n    print("vendored tbr, rewrote", n, "files")\n')
open("harness.py", "w").write('"""Pilot harness: small local LLM agents on tau-bench retail, three guardrail arms.\n\nArms\n  A0  free decoding, native tool-call template (Ollama `tools`), no checks.\n  A1  A0 + post-hoc JSON-schema validation of every tool call, one corrective\n      retry; a call that fails twice is rejected (never executed).\n  A2  grammar-constrained decoding: Ollama `format` = JSON schema of the action\n      union (one branch per tool + a `respond` branch), so every output is a\n      schema-valid action by construction.\n\nThe confirmation gate is not a run arm. It is analysed offline from the traces:\nevery executed write call is what a human confirmer would be shown.\n\nUser side (Amendment 1, 2026-11-18, before the full run): tau-bench\'s own LLM\nuser simulator (same system prompt, verbatim) played by an open-weight model at\ntemperature 0. A scripted user was used in the 9-episode smoke test and dropped,\nbecause many tau-bench instructions encode conditional customer behaviour that a\nfixed reply cannot follow. The episode stops when the simulated user emits\n###STOP###, after `transfer_to_human_agents`, or after MAX_STEPS agent calls.\n\nReward: identical rule to tau-bench `Env.calculate_reward` (hash of the final\ndatabase equals the hash obtained by replaying the gold actions on a fresh copy,\nand every required output string appears in an agent message).\n"""\nimport copy, hashlib, json, re, time, urllib.request\n\nfrom tbr.data import load_data\nfrom tbr.tasks_test import TASKS_TEST\nfrom tbr.tools import ALL_TOOLS\nfrom tbr.tbtypes import Action\n\nWIKI = open(__import__("os").path.join(__import__("os").path.dirname(\n    __import__("tbr").__file__), "wiki.md"), encoding="utf-8").read()\n\nWRITE_TOOLS = {\n    "cancel_pending_order", "exchange_delivered_order_items",\n    "modify_pending_order_address", "modify_pending_order_items",\n    "modify_pending_order_payment", "modify_user_address",\n    "return_delivered_order_items",\n}\nTERMINATE_TOOLS = {"transfer_to_human_agents"}\nMAX_STEPS = 30\n\nRESPOND_DOC = {\n    "type": "function",\n    "function": {\n        "name": "respond",\n        "description": "Send a message to the customer. Use it to ask for confirmation, to give information, or to close the conversation.",\n        "parameters": {\n            "type": "object",\n            "properties": {"content": {"type": "string", "description": "The message to the customer."}},\n            "required": ["content"],\n        },\n    },\n}\nUSER_HELLO = "Hi! How can I help you today?"\n\n\ndef user_system_prompt(instruction):\n    """Verbatim copy of tau-bench LLMUserSimulationEnv.build_system_prompt."""\n    instruction_display = ("\\n\\nInstruction: " + instruction + "\\n") if instruction is not None else ""\n    return f"""You are a user interacting with an agent.{instruction_display}\nRules:\n- Just generate one line at a time to simulate the user\'s message.\n- Do not give away all the instruction at once. Only provide the information that is necessary for the current step.\n- Do not hallucinate information that is not provided in the instruction. For example, if the agent asks for the order id but it is not mentioned in the instruction, do not make up an order id, just say you do not remember or have it.\n- If the instruction goal is satisified, generate \'###STOP###\' as a standalone message without anything else to end the conversation.\n- Do not repeat the exact instruction in the conversation. Instead, use your own words to convey the same information.\n- Try to make the conversation as natural as possible, and stick to the personalities in the instruction."""\n\n\nclass LLMUser:\n    """tau-bench LLM user simulator (roles flipped: the agent speaks as \'user\')."""\n    def __init__(self, client):\n        self.client = client\n        self.calls = 0\n        self.seconds = 0.0\n\n    def _gen(self):\n        res = self.client.chat(self.messages, None, None)\n        self.calls += 1\n        self.seconds += res["_wall"]\n        content = (res.get("message", {}).get("content") or "").strip()\n        self.messages.append({"role": "assistant", "content": content})\n        return content\n\n    def reset(self, instruction):\n        self.messages = [{"role": "system", "content": user_system_prompt(instruction)},\n                         {"role": "user", "content": USER_HELLO}]\n        return self._gen()\n\n    def step(self, agent_text):\n        self.messages.append({"role": "user", "content": agent_text})\n        return self._gen()\n\n\nA2_ADDENDUM = (\n    "\\n\\n# Output format\\nEvery output you produce is one JSON object. To call a tool, "\n    "output {\\"name\\": <tool name>, \\"arguments\\": {...}}. To write to the customer, "\n    "output {\\"name\\": \\"respond\\", \\"arguments\\": {\\"content\\": <your message>}}."\n)\n\nTOOLS_MAP = {t.get_info()["function"]["name"]: t for t in ALL_TOOLS}\nTOOLS_INFO = [t.get_info() for t in ALL_TOOLS]\n\n\n# ---------------------------------------------------------------- environment\ndef _to_hashable(item):\n    if isinstance(item, dict):\n        return tuple((k, _to_hashable(v)) for k, v in sorted(item.items()))\n    if isinstance(item, list):\n        return tuple(_to_hashable(e) for e in item)\n    if isinstance(item, set):\n        return tuple(sorted(_to_hashable(e) for e in item))\n    return item\n\n\ndef data_hash(data):\n    return hashlib.sha256(str(_to_hashable(data)).encode("utf-8")).hexdigest()\n\n\n_BASE_DATA = None\ndef fresh_data():\n    global _BASE_DATA\n    if _BASE_DATA is None:\n        _BASE_DATA = load_data()\n    return copy.deepcopy(_BASE_DATA)\n\n\n_GOLD_HASH = {}\ndef gold_hash(task_id):\n    if task_id not in _GOLD_HASH:\n        data = fresh_data()\n        for a in TASKS_TEST[task_id].actions:\n            if a.name in TOOLS_MAP and a.name not in TERMINATE_TOOLS:\n                try:\n                    TOOLS_MAP[a.name].invoke(data=data, **a.kwargs)\n                except Exception:\n                    pass\n        _GOLD_HASH[task_id] = data_hash(data)\n    return _GOLD_HASH[task_id]\n\n\nclass RetailEnv:\n    def __init__(self, task_id):\n        self.task_id = task_id\n        self.task = TASKS_TEST[task_id]\n        self.data = fresh_data()\n        self.actions = []          # executed actions, tau-bench style\n        self.done = False\n\n    def call(self, name, kwargs):\n        self.actions.append(Action(name=name, kwargs=kwargs))\n        if name in TOOLS_MAP:\n            try:\n                obs = TOOLS_MAP[name].invoke(data=self.data, **kwargs)\n            except Exception as e:\n                obs = f"Error: {e}"\n            if name in TERMINATE_TOOLS:\n                self.done = True\n        else:\n            obs = f"Unknown action {name}"\n        return obs\n\n    def respond(self, content):\n        self.actions.append(Action(name="respond", kwargs={"content": content}))\n\n    def reward(self):\n        r_actions = data_hash(self.data) == gold_hash(self.task_id)\n        outputs = {}\n        for out in self.task.outputs:\n            outputs[out] = any(\n                a.name == "respond" and out.lower() in a.kwargs["content"].lower().replace(",", "")\n                for a in self.actions)\n        r_outputs = all(outputs.values()) if outputs else True\n        return {"reward": float(r_actions and r_outputs), "r_actions": r_actions,\n                "r_outputs": r_outputs, "outputs": outputs}\n\n\n# ---------------------------------------------------------------- schemas\ndef _strict(schema):\n    """Copy of a JSON schema with additionalProperties=false on every object."""\n    s = copy.deepcopy(schema)\n    def walk(x):\n        if isinstance(x, dict):\n            if x.get("type") == "object":\n                x.setdefault("additionalProperties", False)\n                x.setdefault("properties", {})\n            for v in x.values():\n                walk(v)\n        elif isinstance(x, list):\n            for v in x:\n                walk(v)\n    walk(s)\n    return s\n\n\ndef action_union_schema():\n    branches = []\n    for info in TOOLS_INFO + [RESPOND_DOC]:\n        f = info["function"]\n        branches.append({\n            "type": "object",\n            "properties": {"name": {"type": "string", "enum": [f["name"]]},\n                           "arguments": _strict(f["parameters"])},\n            "required": ["name", "arguments"],\n            "additionalProperties": False,\n        })\n    return {"anyOf": branches}\n\n\nPARAM_SCHEMAS = {i["function"]["name"]: _strict(i["function"]["parameters"]) for i in TOOLS_INFO}\n\n\ndef validate_call(name, args):\n    """Post-hoc check used by arm A1. Returns None if valid, else an error string."""\n    import jsonschema\n    if name not in PARAM_SCHEMAS:\n        return f"unknown tool \'{name}\'"\n    if not isinstance(args, dict):\n        return "arguments must be a JSON object"\n    try:\n        jsonschema.validate(args, PARAM_SCHEMAS[name])\n    except jsonschema.ValidationError as e:\n        return f"{e.message} (at {\'/\'.join(str(p) for p in e.path) or \'arguments\'})"\n    return None\n\n\n# ---------------------------------------------------------------- LLM client\nclass OllamaClient:\n    def __init__(self, url, model, num_ctx=16384, seed=7, timeout=600):\n        self.url, self.model, self.num_ctx, self.seed, self.timeout = url, model, num_ctx, seed, timeout\n\n    def chat(self, messages, tools, fmt=None):\n        body = {"model": self.model, "messages": messages, "stream": False,\n                "options": {"temperature": 0, "seed": self.seed, "num_ctx": self.num_ctx,\n                            "num_predict": 1024},\n                "keep_alive": "60m"}\n        if tools:\n            body["tools"] = tools\n        if fmt is not None:\n            body["format"] = fmt\n        req = urllib.request.Request(self.url + "/api/chat", data=json.dumps(body).encode(),\n                                     headers={"Content-Type": "application/json"})\n        t0 = time.time()\n        with urllib.request.urlopen(req, timeout=self.timeout) as r:\n            res = json.loads(r.read())\n        res["_wall"] = time.time() - t0\n        return res\n\n\n# ---------------------------------------------------------------- parsing\nTOOLCALL_HINT = re.compile(r"<tool_call>|^\\s*\\{\\s*\\"name\\"\\s*:", re.S)\n\n\ndef parse_native(msg):\n    """A0/A1: first native tool call, else text. Flags text that looks like a failed call."""\n    tcs = msg.get("tool_calls") or []\n    if tcs:\n        f = tcs[0].get("function", {})\n        args = f.get("arguments", {})\n        if isinstance(args, str):\n            try:\n                args = json.loads(args)\n            except Exception:\n                pass\n        return {"kind": "tool", "name": f.get("name"), "args": args, "malformed_text": False}\n    content = msg.get("content") or ""\n    return {"kind": "text", "content": content, "malformed_text": bool(TOOLCALL_HINT.search(content))}\n\n\ndef parse_constrained(msg):\n    """A2: the content is a JSON action (Ollama may also surface it as a tool call)."""\n    tcs = msg.get("tool_calls") or []\n    if tcs:\n        p = parse_native(msg)\n        if p["name"] == "respond" and isinstance(p["args"], dict):\n            return {"kind": "text", "content": str(p["args"].get("content", "")), "malformed_text": False}\n        return p\n    content = msg.get("content") or ""\n    try:\n        obj = json.loads(content)\n    except Exception:\n        return {"kind": "text", "content": content, "malformed_text": True}\n    name, args = obj.get("name"), obj.get("arguments", {})\n    if name == "respond":\n        return {"kind": "text", "content": str(args.get("content", "")), "malformed_text": False}\n    return {"kind": "tool", "name": name, "args": args, "malformed_text": False}\n\n\n# ---------------------------------------------------------------- episode\ndef run_episode(client, task_id, arm, user_client):\n    env = RetailEnv(task_id)\n    task = env.task\n    system = WIKI + (A2_ADDENDUM if arm == "A2" else "")\n    tools = TOOLS_INFO + ([RESPOND_DOC] if arm == "A2" else [])\n    fmt = action_union_schema() if arm == "A2" else None\n    user = LLMUser(user_client)\n    first = user.reset(task.instruction)\n    messages = [{"role": "system", "content": system},\n                {"role": "user", "content": first}]\n    trace, stats = [], {"llm_calls": 0, "prompt_tokens": 0, "gen_tokens": 0, "llm_seconds": 0.0,\n                        "malformed_text": 0, "invalid_calls": 0, "retries": 0, "rejected": 0,\n                        "env_errors": 0, "user_turns": 0}\n    stop_reason, pending_retry = "max_steps", None\n    dialogue = [{"role": "user", "content": first}]\n    t_start = time.time()\n    for step in range(MAX_STEPS):\n        res = client.chat(messages, tools, fmt)\n        stats["llm_calls"] += 1\n        stats["prompt_tokens"] += res.get("prompt_eval_count", 0) or 0\n        stats["gen_tokens"] += res.get("eval_count", 0) or 0\n        stats["llm_seconds"] += res["_wall"]\n        msg = res.get("message", {})\n        p = parse_constrained(msg) if arm == "A2" else parse_native(msg)\n        rec = {"step": step, "raw_content": (msg.get("content") or "")[:2000],\n               "raw_tool_calls": msg.get("tool_calls"), **{k: v for k, v in p.items()}}\n\n        if p["kind"] == "tool":\n            name, args = p["name"], p["args"]\n            if arm == "A1":\n                err = validate_call(name, args)\n                if err is not None:\n                    stats["invalid_calls"] += 1\n                    rec["guardrail_error"] = err\n                    messages.append({"role": "assistant", "content": "",\n                                     "tool_calls": [{"function": {"name": str(name), "arguments": args if isinstance(args, dict) else {}}}]})\n                    if pending_retry is None:\n                        stats["retries"] += 1\n                        pending_retry = name\n                        note = (f"Guardrail check failed: {err}. The call was NOT executed. "\n                                "Re-issue a corrected call.")\n                        rec["guardrail"] = "retry"\n                    else:\n                        stats["rejected"] += 1\n                        pending_retry = None\n                        note = f"Guardrail check failed again: {err}. The call was rejected and NOT executed."\n                        rec["guardrail"] = "rejected"\n                    messages.append({"role": "tool", "content": note, "tool_name": str(name)})\n                    trace.append(rec)\n                    continue\n                pending_retry = None\n            if not isinstance(args, dict):\n                args = {}\n            obs = env.call(str(name), args)\n            if obs.startswith("Error") or obs.startswith("Unknown action"):\n                stats["env_errors"] += 1\n            rec["observation"] = obs[:1500]\n            rec["write"] = name in WRITE_TOOLS\n            messages.append({"role": "assistant", "content": "",\n                             "tool_calls": [{"function": {"name": str(name), "arguments": args}}]})\n            messages.append({"role": "tool", "content": obs, "tool_name": str(name)})\n            trace.append(rec)\n            if env.done:\n                stop_reason = "transfer"\n                break\n        else:\n            if p["malformed_text"]:\n                stats["malformed_text"] += 1\n            env.respond(p["content"])\n            messages.append({"role": "assistant", "content": p["content"]})\n            trace.append(rec)\n            reply = user.step(p["content"])\n            stats["user_turns"] += 1\n            dialogue += [{"role": "agent", "content": p["content"]}, {"role": "user", "content": reply}]\n            if "###STOP###" in reply:\n                stop_reason = "user_stop"\n                break\n            messages.append({"role": "user", "content": reply})\n    stats["episode_seconds"] = time.time() - t_start\n    stats["user_calls"], stats["user_seconds"] = user.calls, user.seconds\n    rw = env.reward()\n    executed = [a.to_dict() for a in env.actions]\n    return {"task_id": task_id, "arm": arm, "model": client.model, "user_model": user_client.model,\n            **rw, "stop_reason": stop_reason, "dialogue": dialogue,\n            "stats": stats, "executed": executed, "trace": trace,\n            "gold_actions": [a.to_dict() for a in task.actions], "gold_outputs": task.outputs}\n\n\n# ---------------------------------------------------------------- mock client (local tests)\nclass GoldReplayClient:\n    """Replays the gold actions of one task, then answers with the required outputs.\n\n    Used only to test the harness locally without any LLM. `mode` selects the\n    output channel the mock imitates: \'native\' (tool_calls) or \'json\' (A2 content).\n    `corrupt` injects one schema-invalid call first (tests the A1 path).\n    """\n    def __init__(self, task_id, mode="native", corrupt=False):\n        self.model = "gold-replay"\n        acts = [a for a in TASKS_TEST[task_id].actions]\n        self.queue = []\n        if corrupt:\n            self.queue.append(("tool", "get_order_details", {"order": "#W0000000"}))\n        for a in acts:\n            self.queue.append(("tool", a.name, a.kwargs))\n        outs = TASKS_TEST[task_id].outputs\n        self.queue.append(("text", "Done. " + " ".join(outs), None))\n        self.queue.append(("text", "Goodbye.", None))\n        self.mode = mode\n\n    def chat(self, messages, tools, fmt=None):\n        kind, name, args = self.queue.pop(0) if self.queue else ("text", "Goodbye.", None)\n        if self.mode == "json":\n            if kind == "tool":\n                content = json.dumps({"name": name, "arguments": args})\n            else:\n                content = json.dumps({"name": "respond", "arguments": {"content": name}})\n            msg = {"role": "assistant", "content": content}\n        elif kind == "tool":\n            msg = {"role": "assistant", "content": "", "tool_calls": [{"function": {"name": name, "arguments": args}}]}\n        else:\n            msg = {"role": "assistant", "content": name}\n        return {"message": msg, "prompt_eval_count": 0, "eval_count": 0, "_wall": 0.0}\n\n\nclass MockUserClient:\n    """Local test stand-in for the user simulator: two turns, then ###STOP###."""\n    model = "mock-user"\n    def __init__(self):\n        self.n = 0\n    def chat(self, messages, tools, fmt=None):\n        self.n += 1\n        content = "I need help with my order." if self.n == 1 else ("Yes." if self.n < 3 else "###STOP###")\n        return {"message": {"role": "assistant", "content": content}, "_wall": 0.0}\n')
sh(f"{sys.executable} -I vendor_tau.py tau-bench .")


In [ ]:
# 2. Two Ollama servers: GPU 0 serves the agent model, GPU 1 the user simulator.
# NUM_PARALLEL=3 on each, so three episodes run at the same time.
gpus = [l for l in subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")]
if len(gpus) < 2:
    raise RuntimeError("needs GPU T4 x2: agent and user simulator on separate GPUs")
AGENT_PORT, USER_PORT = 11434, 11435
procs = []
for i, port in enumerate((AGENT_PORT, USER_PORT)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), OLLAMA_HOST=f"127.0.0.1:{port}",
               OLLAMA_NUM_PARALLEL="3", OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="60m")
    procs.append(subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open(f"{W}/ollama_{port}.log", "w"), stderr=subprocess.STDOUT))
time.sleep(8)
for m in MODELS + [USER_MODEL]:
    sh(f"OLLAMA_HOST=127.0.0.1:{AGENT_PORT} ollama pull {m}")
sh(f"OLLAMA_HOST=127.0.0.1:{AGENT_PORT} ollama list")
import urllib.request
for m in MODELS + [USER_MODEL]:
    req = urllib.request.Request(f"http://127.0.0.1:{AGENT_PORT}/api/show", data=json.dumps({"model": m}).encode(),
                                 headers={"Content-Type": "application/json"})
    info = json.loads(urllib.request.urlopen(req).read())
    print(m, info.get("details"), flush=True)


In [ ]:
# 3. Campaign (resumable; one JSONL per model)
sys.path.insert(0, W)
import harness as H, threading, queue, traceback
from concurrent.futures import ThreadPoolExecutor
t_campaign = time.time()
def out_path(m): return f"{W}/results_{m.replace(':', '_')}.jsonl"
def done_keys(m):
    p = out_path(m); s = set()
    if os.path.exists(p):
        for line in open(p):
            r = json.loads(line); s.add((r["task_id"], r["arm"]))
    return s
lock = threading.Lock()
def work(m, tid, arm):
    if time.time() - t_campaign > BUDGET_S:
        return "skipped_budget"
    client = H.OllamaClient(f"http://127.0.0.1:{AGENT_PORT}", m, num_ctx=24576)
    user_client = H.OllamaClient(f"http://127.0.0.1:{USER_PORT}", USER_MODEL, num_ctx=8192)
    try:
        r = H.run_episode(client, tid, arm, user_client)
    except Exception as e:
        r = {"task_id": tid, "arm": arm, "model": m, "reward": 0.0, "harness_error": repr(e),
             "tb": traceback.format_exc()[-2000:]}
    with lock:
        open(out_path(m), "a").write(json.dumps(r) + "\n")
    st = r.get("stats", {})
    print(f"{m} task {tid:3d} {arm} reward={r['reward']} stop={r.get('stop_reason')} "
          f"t={st.get('episode_seconds', 0):.0f}s calls={st.get('llm_calls')} user={st.get('user_calls')}", flush=True)
    return r["reward"]
for m, tids in PLAN:
    have = done_keys(m)
    jobs = [(m, t, a) for t in tids for a in ARMS if (t, a) not in have]
    print(f"== {m}: {len(jobs)} episodes to run", flush=True)
    with ThreadPoolExecutor(max_workers=3) as ex:
        list(ex.map(lambda j: work(*j), jobs))
print("campaign wall time (s):", round(time.time() - t_campaign))


In [ ]:
# 4. Quick summary (the real analysis runs offline)
import collections, glob
for p in sorted(glob.glob(f"{W}/results_*.jsonl")):
    rows = [json.loads(l) for l in open(p)]
    agg = collections.defaultdict(lambda: [0, 0])
    for r in rows:
        agg[r["arm"]][0] += r["reward"]; agg[r["arm"]][1] += 1
    print(os.path.basename(p), {a: f"{int(s)}/{n}" for a, (s, n) in sorted(agg.items())},
          "errors:", sum(1 for r in rows if "harness_error" in r))
for p in procs: p.terminate()
